<a href="https://colab.research.google.com/github/mbello126/mbello126/blob/main/train_nids_rf_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Imports and configuration

# Random Forest-based Network Intrusion Detection System (NIDS)

Trains and evaluates a Random Forest classifier on the **NSL-KDD** dataset:
- **Binary model** — Normal vs Attack (primary task)
- **Multi-class model** — Normal / DoS / Probe / R2L / U2R (additional task)

Metrics reported: accuracy, precision, recall, F1-score, ROC-AUC — per objective 4 of Chapter One.

Run the cells top to bottom. The first run downloads `KDDTrain+.txt` / `KDDTest+.txt`
automatically (needs an internet connection) and caches them in `./data/`.
Trained models are saved to `./models/` at the end, ready for the web application.

In [1]:
import os
import urllib.request
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

DATA_DIR = "data"
MODEL_DIR = "models"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_URL = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt"
TEST_URL = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt"

TRAIN_PATH = os.path.join(DATA_DIR, "KDDTrain+.txt")
TEST_PATH = os.path.join(DATA_DIR, "KDDTest+.txt")


## 2. Feature schema and attack-category mapping

The 41 standard NSL-KDD feature names (in order), followed by the `class` (attack label)
and `difficulty_level` columns present in the raw files.

In [2]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "class", "difficulty_level",
]

# Maps every specific NSL-KDD attack label to its attack category.
# The test set contains some attack labels that never appear in training
# (this is intentional in NSL-KDD, to test generalisation).
ATTACK_CATEGORY = {
    # DoS
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "apache2": "DoS", "udpstorm": "DoS",
    "processtable": "DoS", "mailbomb": "DoS", "worm": "DoS",
    # Probe
    "satan": "Probe", "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe",
    "mscan": "Probe", "saint": "Probe",
    # R2L
    "guess_passwd": "R2L", "ftp_write": "R2L", "imap": "R2L", "phf": "R2L",
    "multihop": "R2L", "warezmaster": "R2L", "warezclient": "R2L", "spy": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "snmpguess": "R2L", "snmpgetattack": "R2L",
    "httptunnel": "R2L", "sendmail": "R2L", "named": "R2L",
    # U2R
    "buffer_overflow": "U2R", "loadmodule": "U2R", "rootkit": "U2R",
    "perl": "U2R", "sqlattack": "U2R", "xterm": "U2R", "ps": "U2R",
    # Normal
    "normal": "Normal",
}


## 3. Load the dataset

Downloads the files on first run, then reuses the cached copies in `./data/`.

In [3]:
def download_if_missing(url, path):
    if not os.path.exists(path):
        print(f"Downloading {os.path.basename(path)} ...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"Using cached {path}")


download_if_missing(TRAIN_URL, TRAIN_PATH)
download_if_missing(TEST_URL, TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH, header=None, names=COLUMNS)
test_df = pd.read_csv(TEST_PATH, header=None, names=COLUMNS)

print(train_df.shape, test_df.shape)
train_df.head()


(125973, 43) (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## 4. Preprocessing

- Drop `difficulty_level` (not a feature)
- Map each specific attack label to its category (`Normal` / `DoS` / `Probe` / `R2L` / `U2R`)
- Derive the binary label (`0` = Normal, `1` = Attack)
- Label-encode the three categorical features: `protocol_type`, `service`, `flag`
  (fit on the union of train + test categories so an unseen test-only category doesn't break `transform`)

In [4]:
train_df = train_df.drop(columns=["difficulty_level"]).copy()
test_df = test_df.drop(columns=["difficulty_level"]).copy()

train_df["category"] = train_df["class"].map(ATTACK_CATEGORY)
test_df["category"] = test_df["class"].map(ATTACK_CATEGORY)

unseen = set(test_df.loc[test_df["category"].isna(), "class"].unique())
if unseen:
    print(f"Warning: unmapped attack labels found in test set, dropping those rows: {unseen}")
    test_df = test_df.dropna(subset=["category"])

train_df["binary_label"] = (train_df["category"] != "Normal").astype(int)
test_df["binary_label"] = (test_df["category"] != "Normal").astype(int)

cat_cols = ["protocol_type", "service", "flag"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], test_df[col]]).unique())
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

feature_cols = [c for c in COLUMNS if c not in ("class", "difficulty_level")]

print("Feature count:", len(feature_cols))
train_df["category"].value_counts()


Feature count: 41


,count
category,
Normal,67343
DoS,45927
Probe,11656
R2L,995
U2R,52


## 5. Train the binary model (Normal vs Attack)

In [5]:
X_train, y_train = train_df[feature_cols], train_df["binary_label"]
X_test, y_test = test_df[feature_cols], test_df["binary_label"]

binary_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
binary_model.fit(X_train, y_train)

y_pred = binary_model.predict(X_test)
y_proba = binary_model.predict_proba(X_test)[:, 1]

binary_report = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

for k, v in binary_report.items():
    print(f"{k:12s}: {v:.4f}")

print()
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


accuracy    : 0.7736
precision   : 0.9669
recall      : 0.6237
f1_score    : 0.7583
roc_auc     : 0.9645

              precision    recall  f1-score   support

      Normal       0.66      0.97      0.79      9711
      Attack       0.97      0.62      0.76     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.84      0.77      0.77     22544

Confusion matrix:
 [[9437  274]
 [4829 8004]]


## 6. Train the multi-class model (Normal / DoS / Probe / R2L / U2R)

In [6]:
Xm_train, ym_train = train_df[feature_cols], train_df["category"]
Xm_test, ym_test = test_df[feature_cols], test_df["category"]

multi_model = RandomForestClassifier(
    n_estimators=300, max_depth=None, random_state=42, n_jobs=-1, class_weight="balanced"
)
multi_model.fit(Xm_train, ym_train)

ym_pred = multi_model.predict(Xm_test)
ym_proba = multi_model.predict_proba(Xm_test)

multi_report = {
    "accuracy": accuracy_score(ym_test, ym_pred),
    "precision_macro": precision_score(ym_test, ym_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(ym_test, ym_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(ym_test, ym_pred, average="macro", zero_division=0),
    "roc_auc_ovr_macro": roc_auc_score(
        ym_test, ym_proba, multi_class="ovr", average="macro", labels=multi_model.classes_
    ),
}

for k, v in multi_report.items():
    print(f"{k:20s}: {v:.4f}")

print()
print(classification_report(ym_test, ym_pred, zero_division=0))
print("Classes:", list(multi_model.classes_))
print("Confusion matrix:\n", confusion_matrix(ym_test, ym_pred, labels=multi_model.classes_))


accuracy            : 0.7369
precision_macro     : 0.7411
recall_macro        : 0.4735
f1_macro            : 0.4773
roc_auc_ovr_macro   : 0.9258

              precision    recall  f1-score   support

         DoS       0.96      0.76      0.85      7460
      Normal       0.63      0.97      0.77      9711
       Probe       0.86      0.60      0.70      2421
         R2L       0.75      0.00      0.01      2885
         U2R       0.50      0.03      0.06        67

    accuracy                           0.74     22544
   macro avg       0.74      0.47      0.48     22544
weighted avg       0.78      0.74      0.69     22544

Classes: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
Confusion matrix:
 [[5706 1719   35    0    0]
 [  69 9453  188    0    1]
 [ 163  815 1443    0    0]
 [   0 2864   11    9    1]
 [   0   62    0    3    2]]


## 7. Feature importance (optional, useful for your report)

Shows which of the 41 features the Random Forest relied on most — good material for
your evaluation/discussion chapter.

In [7]:
importances = pd.Series(binary_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(15)


,0
src_bytes,0.175719
dst_bytes,0.114052
flag,0.083533
same_srv_rate,0.079427
dst_host_same_srv_rate,0.075799
dst_host_srv_count,0.055658
logged_in,0.045525
diff_srv_rate,0.044019
dst_host_diff_srv_rate,0.036699
protocol_type,0.035307


## 8. Save the models

Saves both trained models, the three label encoders, and the exact feature-column order
to `./models/` — the web application loads these directly.